# Getting started

> **Install for this tutorial:** `pip install scope-profiler` (base install
> only).

`scope-profiler` measures how long named *regions* of your code take, stores the
raw per-call timings in HDF5, and gives you a Python API and CLI to analyse them.

This notebook covers the recording side:

1. configuring the profiler with `ProfileManager.setup()`
2. marking regions with a context manager and with a decorator
3. writing the results with `ProfileManager.finalize()`
4. a first look at the output file

Everything here runs in a single process — see
[4. Profiling modes](/scope-profiler/guide/04-profiling-modes/) for MPI, LIKWID and the CLI.


In [ ]:
import os
import tempfile

# Work in a scratch directory so the tutorial leaves no files behind.
os.chdir(tempfile.mkdtemp(prefix="scope-profiler-tutorial-"))

In [ ]:
import time

from scope_profiler import ProfileManager

## Configuring the profiler

The `ProfileManager` class methods use a process-wide default manager.
`setup()` picks its recording strategy and output path; instantiate
`ProfileManager()` only when you need independent sessions.

The defaults record wall-clock timings for every call and flush them to disk, so
for most runs `ProfileManager.setup(file_path=...)` is all you need. The optional
`label` names the run wherever several runs have to be told apart: summaries,
chart legends and `scope-profiler inspect`.

Regions entered *before* `setup()` are inert: nothing is recorded until the
profiler is configured, so instrumented code can be imported and run without
profiling it.

In [ ]:
run = ProfileManager.setup(file_path="profiling_data.h5", label="tutorial 1")

`setup()` returns a `ProfilingRun` handle. You rarely need it — `ProfileManager`
itself remembers the configuration — but `run.file_path` is the output path and
`run.finalize()` is the same as `ProfileManager.finalize()`.

## Marking a region with a context manager

`ProfileManager.region(name)` returns a region object you can use as a
context manager. Entering it records a start timestamp, leaving it records the
end — the region is created on first use and reused afterwards.

In [ ]:
def load_data(n):
    time.sleep(0.01)
    return list(range(n))


def transform(values):
    time.sleep(0.005)
    return [value * 2 for value in values]


with ProfileManager.region("load"):
    values = load_data(1000)

for _ in range(3):
    with ProfileManager.region("transform"):
        values = transform(values)

Each `with` block is one *call*. `transform` above was entered three times, so
that region ends up with three recorded durations — the profiler keeps every
call, not just an aggregate.

## Marking a function with a decorator

`@ProfileManager.profile` wraps a whole function. It works with or without
parentheses, and takes an optional region name (the function's `__name__` is
used otherwise).

In [ ]:
@ProfileManager.profile
def solve():
    time.sleep(0.02)


@ProfileManager.profile("assemble_matrix")
def assemble():
    time.sleep(0.008)


for _ in range(2):
    assemble()
    solve()

Decorators may be applied *before* `setup()` is called — at import time, for
instance. `setup()` re-binds every registered decorator to the newly configured
region class, so the decorated function always records with the current
configuration and pays no per-call check for it.

## Nesting regions

Regions nest freely; the profiler records each one independently. Nesting is
what the flame chart in
[3. Visualizing results](/scope-profiler/guide/03-visualization/) reconstructs.

In [ ]:
with ProfileManager.region("timestep"):
    for _ in range(2):
        with ProfileManager.region("timestep.residual"):
            time.sleep(0.004)
        with ProfileManager.region("timestep.update"):
            time.sleep(0.002)

## Finalizing

`finalize()` collects the buffers (under MPI, from every rank onto rank 0),
writes them into one HDF5 file and, by default, prints a per-region summary.
Call it once, at the end of the run. The cell below passes `verbose=False`
because we print the summary ourselves in the next step.

In [ ]:
ProfileManager.finalize(verbose=False)

## Reading the results back

`load()` reads a profile back into a `ProfilingResults`, in whichever format
the file is in (HDF5 or JSON). `print_summary()` is the quickest way to see what
was recorded — all durations are in **seconds**, and nested regions are indented
under their parent. `suppress_notes=True` leaves out the footer of follow-up
commands that `finalize()` prints.

In [ ]:
from scope_profiler import load

results = load("profiling_data.h5")
results.print_summary(suppress_notes=True)

In [ ]:
solve_region = results["solve"]
print(solve_region)
print("calls:", solve_region.num_calls)
print("average duration [s]:", solve_region.average_duration)
print("per-call durations [s]:", solve_region[0].durations)

## What is in the file

The output is a plain HDF5 file, so it is readable with any HDF5 tool.
Timestamps are `int64` nanoseconds, stored as columns shared by all regions
under `events/`. `region_table/names` holds each region name once,
`rank_region_index/` says which slice of the event columns belongs to each
(rank, region) pair, and the run metadata are attributes of the `metadata`
group. The [HDF5 and Python API](/scope-profiler/guide/hdf5-and-python-api/)
guide documents the full layout.

For a quick look at a file without writing any code, `scope-profiler inspect
profiling_data.h5` prints its metadata and per-region statistics.

In [ ]:
import h5py

with h5py.File("profiling_data.h5", "r") as handle:
    handle.visit(print)

By default the file records a *minimal* description of the environment —
versions, platform, CPU model, rank and thread counts, and the `label`. User
and host names, the working directory, environment variables and the like are
only stored with `metadata_detail="full"` (see
[2. Post-processing](/scope-profiler/guide/02-postprocessing/)).

In [ ]:
for key, value in results.metadata.items():
    print(f"{key:>24}: {value}")

## Sessions and the module-level API

`ProfileManager.setup()` plus `finalize()` is the explicit form. A *session* does
both around a block, and finalizes even if the block raises. The same calls are
also available as module-level functions, which saves writing the class name:

In [ ]:
import scope_profiler as sp

with sp.session(file_path="session.h5", label="session demo", verbose=False):
    with sp.region("work"):
        time.sleep(0.01)

load("session.h5").print_summary(suppress_notes=True)

## Next steps

- [2. Post-processing](/scope-profiler/guide/02-postprocessing/) — the analysis API in depth
- [3. Visualizing results](/scope-profiler/guide/03-visualization/) — Gantt, flame and duration charts
- [4. Profiling modes](/scope-profiler/guide/04-profiling-modes/) — configuration, recursive
  profiling, line profiling, MPI, LIKWID and the CLI
- [8. Reports and comparison](/scope-profiler/guide/08-reports-and-comparison/) — HTML reports, `diff`
  and `check`